In [ ]:
!ls -la /content
!ls -la /content/drive
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



In [6]:
import os, sys, time

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA2023_Task2")
    .master("local[*]")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA    = "/content/drive/MyDrive/7006SCN/data/raw"
OUTPUT  = "/content/drive/MyDrive/7006SCN/outputs"
MODELS  = "/content/drive/MyDrive/7006SCN/models"

for p in [OUTPUT, MODELS]:
    os.makedirs(p, exist_ok=True)

CSV_PATH = f"{DATA}/My_Data.csv"

In [7]:
# ── Commit 1: Ingestion + Repartition + Partition Count (CW mandatory) ─────────

from pyspark import StorageLevel
import time

start = time.time()

print("="*50)
print("HMDA 2023 DATA INGESTION")
print("="*50)

# Load dataset
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(CSV_PATH)
)

# Cache immediately after loading
df.persist(StorageLevel.MEMORY_AND_DISK)

# Materialise cache
rows = df.count()
cols = len(df.columns)

load_time = time.time() - start

print(f"Rows               : {rows:,}")
print(f"Columns            : {cols}")
print(f"Load Time          : {load_time:.2f} seconds")
print()

print(f"Default partitions : {df.rdd.getNumPartitions()}")

# Repartition for balanced distributed processing
df = df.repartition(200)

print(f"After repartition  : {df.rdd.getNumPartitions()}")
print()

print(f"Storage Level      : {df.storageLevel}")

print()

print("Spark Version      :", spark.version)
print("Spark UI           :", spark.sparkContext.uiWebUrl)

print()

print("<-- Screenshot this output for Task 2 and Task 4 reports")

HMDA 2023 DATA INGESTION
Rows               : 11,564,178
Columns            : 99
Load Time          : 544.03 seconds

Default partitions : 33
After repartition  : 200

Storage Level      : Serialized 1x Replicated

Spark Version      : 4.0.3
Spark UI           : http://4fd55f385307:4040

<-- Screenshot this output for Task 2 and Task 4 reports


In [ ]:
print(f"Estimated rows/partition : {rows//df.rdd.getNumPartitions():,}")

In [ ]:
# ── Commit 2: Data Quality Assessment + Cleaning (CW mandatory) ────────────────
# Commit 2: feat(task2): perform data quality assessment and preprocessing

# Dataset Overview
import time
from pyspark.sql import functions as F

start = time.time()

print("="*80)
print("HMDA 2023 DATA QUALITY ASSESSMENT")
print("="*80)

print(f"Rows                 : {df.count():,}")
print(f"Columns              : {len(df.columns)}")
print(f"Partitions           : {df.rdd.getNumPartitions()}")
print()

# Data Types Summary
print("="*50)
print("COLUMN DATA TYPES")
print("="*50)

type_summary = (
    df.dtypes
)

for name, dtype in type_summary:
    print(f"{name:<40} {dtype}")
# Missing Value Analysis
print("\n" + "="*50)
print("MISSING VALUE ANALYSIS")
print("="*50)

rows = df.count()

missing = []

for c in df.columns:
    col_type = dict(df.dtypes)[c]

    if col_type == "string":
        # For string columns, check for nulls or empty strings
        null_condition = F.col(c).isNull() | (F.col(c) == "")
    elif col_type in ["double", "float"]:
        # For float/double columns, check for nulls or NaN
        null_condition = F.col(c).isNull() | F.isnan(c)
    else:
        # For other numeric types (int, bigint, etc.), only check for nulls.
        # Comparing an int/bigint column to "" will cause the casting error.
        null_condition = F.col(c).isNull()

    nulls = df.filter(null_condition).count()

    missing.append((
        c,
        nulls,
        round((nulls/rows)*100,2)
    ))

missing_df = (
    spark.createDataFrame(
        missing,
        ["Feature","Missing Count","Missing %"]
    )
    .orderBy(F.desc("Missing %"))
)

missing_df.show(99, truncate=False)
# Duplicate Analysis

print("\n" + "="*50)
print("DUPLICATE ANALYSIS")
print("="*50)

duplicate_rows = rows - df.dropDuplicates().count()

print(f"Duplicate Records : {duplicate_rows:,}")
print(f"Duplicate %       : {(duplicate_rows/rows)*100:.4f}%")
# Target Distribution
print("\n" + "="*50)
print("TARGET DISTRIBUTION")
print("="*50)

(
df.groupBy("action_taken")
.count()
.orderBy("action_taken")
.show(truncate=False)
)
# Data Leakage Assessment
print("\n" + "="*50)
print("DATA LEAKAGE ASSESSMENT")
print("="*50)

potential_leakage = [

    "action_taken",

    "denial_reason_1",

    "denial_reason_2",

    "denial_reason_3",

    "denial_reason_4"

]

print("Potential leakage variables:")

for col in potential_leakage:
    if col in df.columns:
        print(f"✓ {col}")

# Numeric Summary
print("\n" + "="*50)
print("NUMERIC FEATURE SUMMARY")
print("="*50)

numeric = [

c for c,t in df.dtypes

if t in ("int","bigint","double","float","long")

]

df.select(numeric).describe().show()

print("""
================ PREPROCESSING STRATEGY ================

Missing Values
--------------
• Numeric features -> Median Imputation
• Categorical features -> 'Unknown'

Duplicates
----------
• Remove duplicate applications if present

Outliers
--------
• Detect using IQR
• Retain unless clearly invalid

Encoding
--------
• StringIndexer
• OneHotEncoder

Scaling
-------
• StandardScaler (for Logistic Regression)

========================================================
""")


In [ ]:
# Processing Time
end = time.time()

print("="*80)
print(f"Commit 2 Execution Time : {(end-start):.2f} seconds")
print("="*80)

Commit 2 Execution Time : 2759.39 seconds


In [ ]:

# Commit 3: HMDA Data Cleaning + Feature Engineering
# feat(task2): HMDA preprocessing and feature engineering


from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time

start = time.time()

print("="*80)
print("COMMIT 3 : HMDA DATA CLEANING + FEATURE ENGINEERING")
print("="*80)


# STEP 1:  Work on a clean copy


df_clean = df


# STEP 2: Create Safe Numeric Columns


from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

def safe_to_double(col_name):
    """
    Robust conversion for HMDA numeric fields.
    Handles:
    NA
    Exempt
    blanks
    spaces
    commas
    """
    return (
        F.when(
            F.trim(F.col(col_name)).isin("NA", "Exempt", "", " "),
            None
        )
        .otherwise(
            F.regexp_replace(
                F.trim(F.col(col_name)),
                "[^0-9.]",
                ""
            )
        )
        .cast(DoubleType())
    )

df_clean = (
    df
    .withColumn("income_num", safe_to_double("income"))
    .withColumn("property_value_num", safe_to_double("property_value"))
    .withColumn("interest_rate_num", safe_to_double("interest_rate"))
    .withColumn("loan_term_num", safe_to_double("loan_term"))
    .withColumn("ltv_num", safe_to_double("combined_loan_to_value_ratio"))
)


# STEP 3: Clean Categorical Values


df_clean = (
    df_clean
    .withColumn(
        "applicant_age",
        F.when(
            F.col("applicant_age") == "8888",
            None
        ).otherwise(F.col("applicant_age"))
    )
    .withColumn(
        "debt_to_income_ratio",
        F.when(
            F.col("debt_to_income_ratio").isin("NA", "Exempt"),
            "Unknown"
        ).otherwise(F.col("debt_to_income_ratio"))
    )
)

# STEP 4: Median Imputation


numeric_cols = [
    "income_num",
    "property_value_num",
    "interest_rate_num",
    "loan_term_num",
    "ltv_num"
]

for c in numeric_cols:

    non_null = df_clean.filter(F.col(c).isNotNull())

    median = non_null.approxQuantile(c, [0.5], 0.01)

    if len(median) > 0:
        df_clean = df_clean.fillna({c: median[0]})


# ── Additional Numeric Columns ────────────────────────────────────────────────

numeric_conversion = [
    ("loan_amount", "loan_amount_num"),
    ("tract_to_msa_income_percentage", "msa_income_pct"),
    ("tract_minority_population_percent", "minority_pct"),
    ("tract_owner_occupied_units", "owner_units"),
    ("ffiec_msa_md_median_family_income", "msa_family_income")
]

for source, target in numeric_conversion:

    df_clean = df_clean.withColumn(
        target,
        safe_to_double(source)
    )
df_clean.select(
    "loan_amount_num",
    "msa_income_pct",
    "minority_pct",
    "owner_units",
    "msa_family_income"
).printSchema()
df_clean.select(
    "income_num",
    "property_value_num",
    "interest_rate_num",
    "loan_term_num",
    "ltv_num"
).printSchema()

# ── Compute Quartiles for Feature Engineering ────────────────────────────────

loan_q = df_clean.approxQuantile(
    "loan_amount_num",
    [0.25, 0.50, 0.75],
    0.01
)

income_q = df_clean.approxQuantile(
    "income_num",
    [0.25, 0.50, 0.75],
    0.01
)

property_q = df_clean.approxQuantile(
    "property_value_num",
    [0.25, 0.50, 0.75],
    0.01
)

interest_q = df_clean.approxQuantile(
    "interest_rate_num",
    [0.25, 0.50, 0.75],
    0.01
)

print("Loan Amount Quartiles :", loan_q)
print("Income Quartiles      :", income_q)
print("Property Quartiles    :", property_q)
print("Interest Quartiles    :", interest_q)

# STEP 5: HMDA Feature Engineering


# -------------------------------------------------------------------------
# 1. Loan-to-Income Ratio
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "loan_income_ratio",
    F.when(
        F.col("income_num") > 0,
        F.col("loan_amount_num") / F.col("income_num")
    ).otherwise(0.0)
)

# -------------------------------------------------------------------------
# 2. Loan Amount Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "loan_size",
    F.when(F.col("loan_amount_num") < 105000, "Low")
     .when(F.col("loan_amount_num") < 355000, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 3. Income Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "income_band",
    F.when(F.col("income_num") < 69, "Low")
     .when(F.col("income_num") < 142, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 4. Property Value Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "property_band",
    F.when(F.col("property_value_num") < 275000, "Low")
     .when(F.col("property_value_num") < 485000, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 5. Interest Rate Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "interest_band",
    F.when(F.col("interest_rate_num") < 6.50, "Low")
     .when(F.col("interest_rate_num") < 7.00, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 6. Loan-to-Value Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "ltv_band",
    F.when(F.col("ltv_num") < 80, "Low")
     .when(F.col("ltv_num") < 95, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 7. Owner Occupied
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "owner_occupied",
    F.when(F.col("occupancy_type") == "1", 1.0)
     .otherwise(0.0)
)

# -------------------------------------------------------------------------
# 8. First Lien
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "first_lien",
    F.when(F.col("lien_status") == "1", 1.0)
     .otherwise(0.0)
)

# -------------------------------------------------------------------------
# 9. Jumbo Loan
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "jumbo_loan",
    F.when(F.col("conforming_loan_limit") == "NC", 1.0)
     .otherwise(0.0)
)

# -------------------------------------------------------------------------
# 10. Metro Income Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "metro_income_band",
    F.when(F.col("msa_income_pct") < 80, "Low")
     .when(F.col("msa_income_pct") < 120, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 11. Minority Population Band
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "minority_band",
    F.when(F.col("minority_pct") < 25, "Low")
     .when(F.col("minority_pct") < 60, "Medium")
     .otherwise("High")
)

# -------------------------------------------------------------------------
# 12. Existing HMDA Categories
# -------------------------------------------------------------------------

df_clean = (
    df_clean
    .withColumn("dti_category", F.col("debt_to_income_ratio"))
    .withColumn("age_group", F.col("applicant_age"))
)

print("=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)

engineered = [
    "loan_income_ratio",
    "loan_size",
    "income_band",
    "property_band",
    "interest_band",
    "ltv_band",
    "owner_occupied",
    "first_lien",
    "jumbo_loan",
    "metro_income_band",
    "minority_band",
    "dti_category",
    "age_group"
]

for feature in engineered:
    print("✓", feature)

print("\nTotal engineered features:", len(engineered))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Commit 4 : Spark ML Pipeline Construction
# feat(task2): HMDA preprocessing pipeline
# ─────────────────────────────────────────────────────────────────────────────

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

print("="*80)
print("COMMIT 4 : SPARK ML PIPELINE")
print("="*80)

###########################################################################
# TARGET
###########################################################################

TARGET = "loan_approval"
# ──────────────────────────────────────────────────────────────────────────────
# Commit 4 : Binary Target Construction
# feat(task2): create loan_approval target
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.sql import functions as F

print("="*80)
print("CREATING BINARY TARGET")
print("="*80)

# -------------------------------------------------------------------------
# HMDA Binary Target
# 1 = Approved
# 0 = Not Approved
# -------------------------------------------------------------------------

df_clean = df_clean.withColumn(
    "loan_approval",
    F.when(
        F.col("action_taken").isin("1", "2"),
        1
    ).otherwise(0)
)

df_clean.groupBy("loan_approval") \
    .count() \
    .orderBy("loan_approval") \
    .show()

print("="*80)
print("Target Created Successfully")
print("="*80)
# Remove Leakage Columns
LEAKAGE_COLUMNS = [
    "action_taken",
    "denial_reason_1",
    "denial_reason_2",
    "denial_reason_3",
    "denial_reason_4"
]

df_model = df_clean.drop(*LEAKAGE_COLUMNS)

print("Leakage columns removed ✓")

# Define Features
NUM_COLS = [

    "loan_amount_num",

    "income_num",

    "property_value_num",

    "interest_rate_num",

    "loan_term_num",

    "ltv_num",

    "loan_income_ratio",

    "owner_units",

    "msa_income_pct",

    "minority_pct",

    "msa_family_income",

    "owner_occupied",

    "first_lien",

    "jumbo_loan"

]

# Categorical Features
CAT_COLS = [

    "loan_size",

    "income_band",

    "property_band",

    "interest_band",

    "ltv_band",

    "metro_income_band",

    "minority_band",

    "dti_category",

    "age_group",

    "derived_race",

    "derived_ethnicity",

    "derived_sex",

    "loan_type",

    "loan_purpose",

    "occupancy_type"

]

# Creating Indexers
indexers = [

    StringIndexer(

        inputCol=c,

        outputCol=c+"_idx",

        handleInvalid="keep"

    )

    for c in CAT_COLS

]
# One Hot Encoder
encoders = [

    OneHotEncoder(

        inputCol=c+"_idx",

        outputCol=c+"_vec"

    )

    for c in CAT_COLS

]

# Feature Assembly
feature_columns = NUM_COLS + [

    c+"_vec"

    for c in CAT_COLS

]

assembler = VectorAssembler(

    inputCols=feature_columns,

    outputCol="features_raw",

    handleInvalid="keep"

)

# Feature Scaling
scaler = StandardScaler(

    inputCol="features_raw",

    outputCol="features",

    withMean=False,

    withStd=True

)

# Building Pipeline
pipeline = Pipeline(

    stages=

        indexers +

        encoders +

        [

            assembler,

            scaler

        ]

)
# Fit Pipeline
print("="*80)
print("FITTING PIPELINE")
print("="*80)

pipeline_model = pipeline.fit(df_model)

df_model = pipeline_model.transform(df_model)
# Verification
print("="*80)
print("PIPELINE COMPLETE")
print("="*80)

df_model.select(

    TARGET,

    "features"

).show(5, truncate=False)
pipeline_model.write().overwrite().save(

    f"{MODELS}/preprocessing_pipeline"

)

print("Pipeline saved ✓")